# Train a hand-drawn flowchart detector (YOLOv11)

Trains a YOLO model to detect flowchart **shapes + arrows** on the free, open
`handwritten-diagram-datasets` (FC_A / FC_B / hdBPMN ...), then lets you test it
on your own photo and download `best.pt`.

**Before running:** `Runtime` -> `Change runtime type` -> **GPU (T4)**. Then
`Runtime` -> `Run all`. Training is ~30-60 min. At the end you'll upload your
notes photo to see live detections, and download `best.pt`.


In [ ]:
!pip -q install ultralytics pyyaml
import torch
print('CUDA available:', torch.cuda.is_available())


In [ ]:
# Download the free, open dataset (COCO format, no signup).
!git clone --depth 1 https://github.com/bernhardschaefer/handwritten-diagram-datasets /content/hdd
import glob
jsons = glob.glob('/content/hdd/**/*.json', recursive=True)
print(len(jsons), 'json files found')
for j in jsons:
    print(' ', j)


In [ ]:
# Convert all COCO files into a single unified YOLO dataset.
import json, glob, os, shutil, random
from pathlib import Path
from collections import defaultdict
random.seed(0)

SRC = '/content/hdd'
OUT = Path('/content/yolo_ds')
for s in ['train', 'val']:
    (OUT/'images'/s).mkdir(parents=True, exist_ok=True)
    (OUT/'labels'/s).mkdir(parents=True, exist_ok=True)

coco_files = glob.glob(SRC + '/**/*.json', recursive=True)
valid, cat_names = [], set()
for cf in coco_files:
    try:
        d = json.load(open(cf))
    except Exception:
        continue
    if not (isinstance(d, dict) and 'images' in d and 'annotations' in d and 'categories' in d):
        continue
    valid.append((cf, d))
    for c in d['categories']:
        cat_names.add(c['name'])

classes = sorted(cat_names)
cls_idx = {n: i for i, n in enumerate(classes)}
print('unified classes:', classes)

# Index every image on disk by basename so we can resolve COCO file_name fields.
disk = {}
for p in glob.glob(SRC + '/**/*', recursive=True):
    if p.lower().endswith(('.jpg', '.jpeg', '.png')):
        disk.setdefault(os.path.basename(p), p)

written = 0
for cf, d in valid:
    id2name = {c['id']: c['name'] for c in d['categories']}
    by_img = defaultdict(list)
    for a in d['annotations']:
        by_img[a['image_id']].append(a)
    tag = Path(cf).stem
    for im in d['images']:
        base = os.path.basename(im['file_name'])
        src = disk.get(base)
        if not src:
            continue
        W, H = im.get('width'), im.get('height')
        if not W or not H:
            continue
        # Random 90/10 split (the datasets' official test sets are huge and would
        # waste ~half the data on slow per-epoch validation).
        split = 'val' if random.random() < 0.10 else 'train'
        uniq = tag + '__' + base
        dst = OUT/'images'/split/uniq
        if not dst.exists():
            shutil.copy(src, dst)
        lines = []
        for a in by_img[im['id']]:
            name = id2name.get(a['category_id'])
            if name not in cls_idx:
                continue
            x, y, w, h = a['bbox']
            if w <= 0 or h <= 0:
                continue
            cx, cy = (x + w/2)/W, (y + h/2)/H
            lines.append(str(cls_idx[name]) + ' ' + format(cx, '.6f') + ' ' + format(cy, '.6f') + ' ' + format(w/W, '.6f') + ' ' + format(h/H, '.6f'))
        (OUT/'labels'/split/(Path(uniq).stem + '.txt')).write_text('\n'.join(lines))
        written += 1

print('images written:', written)
import yaml
names = {i: nm for nm, i in cls_idx.items()}
yaml.safe_dump({'path': str(OUT), 'train': 'images/train', 'val': 'images/val', 'names': names}, open(OUT/'data.yaml', 'w'))
print(open(OUT/'data.yaml').read())


In [ ]:
# Train. imgsz=1024 helps small arrows/labels; lower batch if you hit OOM.
from ultralytics import YOLO
model = YOLO('yolo11s.pt')
model.train(data='/content/yolo_ds/data.yaml', epochs=100, imgsz=1024,
            batch=8, patience=25, project='/content/runs', name='flow')


In [ ]:
# Sanity-check: detections on a few validation images.
import glob
from PIL import Image as PImage
from IPython.display import display
for ip in sorted(glob.glob('/content/yolo_ds/images/val/*'))[:4]:
    r = model.predict(ip, imgsz=1024, conf=0.25, verbose=False)
    display(PImage.fromarray(r[0].plot()[:, :, ::-1]))


## Test on YOUR photo (the real GO/NO-GO)
Run the next cell and upload `sample_diagram.jpg` (your notes photo). This is the
honest test of whether the model generalizes to a cluttered phone photo.


In [ ]:
from google.colab import files
from PIL import Image as PImage
from IPython.display import display
up = files.upload()
name = list(up.keys())[0]
r = model.predict(name, imgsz=1024, conf=0.2, verbose=False)
display(PImage.fromarray(r[0].plot()[:, :, ::-1]))
print('detections:')
for b in r[0].boxes:
    print(' ', model.names[int(b.cls)], round(float(b.conf), 2))


In [ ]:
# Download the trained weights -> save as backend/models/flowchart.pt in your repo.
from google.colab import files
files.download('/content/runs/flow/weights/best.pt')
